In [1]:
import locale
locale.getpreferredencoding = lambda: "UTF-8"

In [2]:
!pip install json_repair

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 kB 1.1 MB/s eta 0:00:00a 0:00:01


In [3]:
!pip install pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.9/68.9 MB 26.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 68.8 MB/s eta 0:00:00:00:0100:01


In [ ]:
import os
import re
import io
import sys
import json
import time
import locale

locale.getpreferredencoding = lambda: "UTF-8"
os.environ["PYTHONIOENCODING"] = "utf-8"
os.environ["PYTHONUTF8"] = "1"

import pymupdf
from PIL import Image
from typing import List, Optional, Literal, Union
from pydantic import BaseModel, Field, field_validator, model_validator
from google import genai
from google.genai import types
from json_repair import repair_json


# 1. SHARED SYSTEM TAXONOMY
VehicleType = Literal[
    "PASSENGER CAR", "TRUCK", "MULTIPURPOSE PASSENGER VEHICLE (MPV)",
    "BUS", "INCOMPLETE VEHICLE", "OTHER"
]

BodyClass = Literal[
    "Cargo Van", "Convertible/Cabriolet", "Coupe", "Crossover Utility Vehicle (CUV)",
    "Hatchback/Liftback/Notchback", "Incomplete", "Incomplete - Chassis Cab (Number of Cab Unknown)",
    "Incomplete - Chassis Cab (Single Cab)", "Incomplete - Cutaway", "Incomplete - Motor Home Chassis",
    "Incomplete - Stripped Chassis", "Minivan", "Pickup", "Roadster", "Sedan/Saloon",
    "Sport Utility Truck (SUT)", "Sport Utility Vehicle (SUV)/Multi-Purpose Vehicle (MPV)",
    "Truck", "Van", "Wagon", "Other"
]

CarPart = Literal[
    "Bumper", "Front Bumper", "Rear Bumper", "Bumper Reinforcement", "Bumper Support",
    "Bumper Fascia", "Lower Valance", "Hood", "Grille", "Headlight", "Taillight",
    "Fender", "Quarter Panel", "Front Door", "Rear Door", "Door", "Door Handle", "Door Jamb", "Side Mirror",
    "Windshield", "Side Window", "Rear Window", "Windshield Header", "Rear Window Header",
    "Rear Window Ledge", "Roof", "Roof Rail", "Roof Header", "Roof Rack", "Headliner",
    "Rocker Panel", "Molding", "Weatherstrip", "A-Pillar", "B-Pillar", "C-Pillar", "D-Pillar", "Other Pillar",
    "Trunk Lid", "Rear Hatch", "Tailgate", "Spoiler", "Body Trim", "Running Board",
    "Truck Cab", "Pickup Bed", "Bedside Panel", "Bed Rail", "Vehicle Canopy",
    "Wheel", "Tire", "Rim", "Wheel Cover", "Wheel Hub", "Wheel Well", "Axle",
    "Suspension", "Control Arm", "Differential", "Drivetrain", "Frame", "Frame Rail",
    "Crossmember", "Radiator", "Radiator Support", "Radiator Mounting Bracket",
    "Skid Plate", "Underbody", "Seat", "License Plate", "Trailer Hitch", "Park Sensor", "Sensor Cap", "Radar", "Other"
]

OperationType = Literal["Replace", "Repair", "Refinish", "Blend", "Not Specified", "Exclude"]

ALLOWED_CAR_PARTS = set(CarPart.__args__)


# 2. SCHEMAS 
class ExtractedCarInfo(BaseModel):
    car_make: Optional[str] = Field(None, description="Vehicle Make (e.g. Toyota, Dodge).")
    car_model: Optional[str] = Field(None, description="Vehicle Model (e.g. Camry, Ram).")
    car_model_year: Optional[int] = Field(None, description="4-digit Model Year.")
    vehicle_type: VehicleType = Field("PASSENGER CAR", description="Mapped Vehicle Type enum.")
    body_class: BodyClass = Field("Sedan/Saloon", description="Mapped Body Class enum.")

class ExtractedRepairedPart(BaseModel):
    original_description: str = Field(..., description="EXACT line item text printed on the invoice.")
    standard_part: CarPart = Field(..., description="Mapped part corresponding directly to system CarPart taxonomy.")
    operation: OperationType = Field(..., description="Action code: Replace, Repair, Refinish, or Blend.")

    @field_validator('operation', mode='before')
    @classmethod
    def parse_and_validate_operation(cls, v: str) -> str:
        v_clean = str(v).strip().lower()
        if any(x in v_clean for x in ["r&i", "r/i", "rbi", "rai", "remove"]):
            return "Exclude"
        if any(x in v_clean for x in ["repl", "replace", "repi", "rep/l"]):
            return "Replace"
        elif any(x in v_clean for x in ["blnd", "blend", "bind"]):
            return "Blend"
        elif any(x in v_clean for x in ["rpr", "repair"]):
            return "Repair"
        elif any(x in v_clean for x in ["refn", "refinish", "paint"]):
            return "Refinish"
        else:
            return "Exclude"

    @field_validator('standard_part', mode='before')
    @classmethod
    def auto_fix_part_typos(cls, v: str) -> str:
        if not isinstance(v, str) or not v.strip():
            return "Other"
        
        v_clean = v.strip()
        typo_map = {
            "tailight": "Taillight", "tail light": "Taillight",
            "head light": "Headlight", "front bumper": "Front Bumper",
            "rear bumper": "Rear Bumper", "door shell": "Door",
            "rocker panel": "Rocker Panel", "quarter panel": "Quarter Panel",
            "rocker molding": "Rocker Panel", "stone guard": "Body Trim",
            "door w/strip": "Door Jamb", "belt molding": "Body Trim",
            "auto park sensor": "Park Sensor", 
            "park sensor": "Park Sensor", 
            "reverse sensor": "Park Sensor",
            "reverse sensor cap": "Sensor Cap",
            "sensor cap": "Sensor Cap"
        }
        
        if v_clean.lower() in typo_map:
            return typo_map[v_clean.lower()]
            
        formatted = v_clean.title()
        if formatted in ALLOWED_CAR_PARTS:
            return formatted
        return "Other"

class CleanInvoiceExtraction(BaseModel):
    car_info: ExtractedCarInfo = Field(..., description="Header vehicle meta information extracted from the invoice.")
    repaired_parts: List[ExtractedRepairedPart] = Field(default_factory=list, description="List of physical repaired/replaced/blended parts.")

    @model_validator(mode='after')
    def strict_labor_and_supplies_filter(self):
        filtered_list = []
        section_headers = ["rear door", "front door", "quarter panel", "rear lamps", "pillars", "rear bumper", "pillars, rocker & floor"]
        blacklisted_terms = [
            "coat", "spray", "mask", "sand", "reset", "scan", "waste", "hazardous",
            "calibrate", "labor", "buff", "undercoat", "deadner", "oil", "fluid",
            "washer", "cleaner", "coolant", "spark plug", "grease", "chemical",
            "zip tie", "filter", "shim", "seal kit", "syn", "synthetic", "drum",
            "75w", "5w40", "dot4", "r&i", "r/i", "subl", "pre and post scan", "clear coat"
        ]
        seen_descriptions = set()

        for part in self.repaired_parts:
            desc_clean = part.original_description.strip().lower()

            if part.operation == "Exclude":
                continue
            if desc_clean in section_headers:
                continue
            if any(term in desc_clean for term in blacklisted_terms):
                continue
            if desc_clean in seen_descriptions:
                continue
            
            seen_descriptions.add(desc_clean)
            filtered_list.append(part)
            
        self.repaired_parts = filtered_list
        return self

SCHEMA_STR = json.dumps(CleanInvoiceExtraction.model_json_schema(), indent=2)

SMART_INVOICE_PROMPT = f"""
You are a Spatial OCR Document Engine for Automotive Estimates.

CRITICAL EXECUTION LOGIC (STRICT STEP-BY-STEP AUDITING):

STEP 1: IDENTIFY TABLE COLUMNS
Locate the table header: [Line] [Oper] [Description] ...
The 'Oper' column is strictly positioned between [Line] and [Description].

STEP 2: ROW-BY-ROW COLUMN AUDIT (DO NOT SKIP ANY CHARACTERS):
For EVERY numbered row, extract the EXACT raw text inside the 'Oper' column bounds.
Apply the following strict filters:

A. EMPTY/BLANK Oper Cell:
    - If the cell under 'Oper' is empty, space, or missing (e.g., "O/H bumper assy"): DISCARD ROW IMMEDIATELY. Do not invent an operation.

B. ALLOWED OPER VALUES (Exact Match):
    - 'Repl' or 'Repi' -> Map operation to "Replace"
    - 'Rpr' or 'Repair' -> Map operation to "Repair"
    - 'Blnd' or 'Bind' -> Map operation to "Blend"
    - 'Refn' or 'Paint' -> Map operation to "Refinish"

C. STRICT DISCARD VALUES:
    - 'R&I', 'R/I', 'RBI', 'RAI', 'Subl' -> DISCARD ROW IMMEDIATELY.

STEP 3: CATEGORY & DESCRIPTION CLEANUP
- Ignore section header rows (e.g., REAR BUMPER, VEHICLE DIAGNOSTICS).
- Do NOT discard body parts that are being refinished or blended (e.g. "Blend rt. roof rail" is VALID).
- IGNORE paint materials/supplies and fees (e.g., Pre-repair scan, Post-repair scan, Clear coat, Flex additive, Hazardous waste).
- Extract physical sensors (e.g., Reverse sensor, Park sensor, Radar) as valid parts under 'Park Sensor' or 'Sensor Cap'.

Return ONLY valid raw JSON matching {SCHEMA_STR}. Do not include any extra introductory text.
"""

class InvoiceModel:
    def __init__(self, api_keys: List[str], model_name: str = "gemini-3.6-flash"):
        self.api_keys = api_keys
        self.current_key_idx = 0
        self.model_name = model_name
        self.prompt = SMART_INVOICE_PROMPT
        self.client = None
        self.load_models()

    def load_models(self):
        current_key = self.api_keys[self.current_key_idx]
        os.environ["GEMINI_API_KEY"] = current_key
        self.client = genai.Client(api_key=current_key)

    def switch_to_next_key(self):
        if len(self.api_keys) > 1:
            self.current_key_idx = (self.current_key_idx + 1) % len(self.api_keys)
            print(f"🔄 Switching to API Key Index: {self.current_key_idx + 1}")
            self.load_models()

    def load_document_as_images(self, file_path: str, dpi: int = 200) -> List[Image.Image]:
        if not os.path.exists(file_path):
            raise FileNotFoundError(f"Document not found at path: {file_path}")

        ext = os.path.splitext(file_path)[1].lower()
        images = []

        print("=" * 70)
        print(f"PROCESSING DOCUMENT: {os.path.basename(file_path)}")
        print("=" * 70)

        if ext == ".pdf":
            doc = pymupdf.open(file_path)
            for page_idx in range(len(doc)):
                page = doc[page_idx]
                pix = page.get_pixmap(dpi=dpi, alpha=False)
                img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
                images.append(img)
                print(f"  --> Extracted Page {page_idx + 1}")
            doc.close()
        elif ext in [".png", ".jpg", ".jpeg", ".webp"]:
            img = Image.open(file_path).convert("RGB")
            images.append(img)
            print("  --> Image Loaded Successfully")
        else:
            raise ValueError("Unsupported file extension.")
            
        return images

    @staticmethod
    def _extract_json(raw_text: str) -> dict:
        match = re.search(r'```(?:json)?\s*([\s\S]*?)\s*```', raw_text)
        text = match.group(1).strip() if match else raw_text.strip()
        
        try:
            return json.loads(text)
        except Exception:
            pass

        try:
            repaired_string = repair_json(text)
            return json.loads(repaired_string)
        except Exception:
            cleaned = re.sub(r',\s*([\}])', r'\1', text)
            cleaned = re.sub(r"(?<!\\)'", '"', cleaned)
            return json.loads(cleaned)

    def extract_information(self, invoice: Union[str, List[Image.Image]], dpi: int = 200) -> str:
        images = self.load_document_as_images(invoice, dpi=dpi) if isinstance(invoice, str) else invoice

        contents = []
        for img in images:
            img_byte_arr = io.BytesIO()
            img.save(img_byte_arr, format='JPEG', quality=95)
            contents.append(types.Part.from_bytes(data=img_byte_arr.getvalue(), mime_type="image/jpeg"))

        contents.append(self.prompt)

        max_attempts = 4
        for attempt in range(max_attempts):
            try:
                print(f"RUNNING {self.model_name.upper()} VISION INFERENCE (Attempt {attempt+1})...")
                response = self.client.models.generate_content(
                    model=self.model_name,
                    contents=contents,
                    config=types.GenerateContentConfig(
                        temperature=0.0,
                        response_mime_type="application/json"
                    )
                )

                output_text = response.text
                raw_json = self._extract_json(output_text)
                validated_output = CleanInvoiceExtraction(**raw_json)
                return json.dumps(validated_output.model_dump(), indent=2, ensure_ascii=False)

            except Exception as e:
                err_str = str(e)
                if "429" in err_str:
                    print(f"⚠️ Limit 429 encountered. Switching API Key...")
                    self.switch_to_next_key()
                    time.sleep(2)
                elif "503" in err_str:
                    wait_time = (attempt + 1) * 4
                    print(f"⏳ Server Busy 503. Retrying in {wait_time}s...")
                    time.sleep(wait_time)
                else:
                    if attempt == max_attempts - 1:
                        raise e
                    time.sleep(3)

In [5]:
import os
import time
import json
from typing import Dict, List, Any

class InvoicePipelineEvaluator:
    def __init__(self, model: InvoiceModel):
        self.model = model

    def calculate_part_metrics(self, pred_parts: List[Dict], gt_parts: List[Dict]) -> Dict[str, float]:
        pred_set = {(p["standard_part"], p["operation"]) for p in pred_parts}
        gt_set = {(p["standard_part"], p["operation"]) for p in gt_parts}

        true_positives = len(pred_set.intersection(gt_set))
        false_positives = len(pred_set - gt_set)
        false_negatives = len(gt_set - pred_set)

        precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0.0
        recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0.0
        f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

        return {"precision": precision, "recall": recall, "f1_score": f1}

    def evaluate_dataset(self, dataset_dir: str, ground_truth_dir: str) -> Dict[str, Any]:
        results = {
            "total_documents": 0,
            "schema_compliant_count": 0,
            "total_latency_sec": 0.0,
            "header_exact_matches": 0,
            "part_f1_scores": [],
            "part_precisions": [],
            "part_recalls": []
        }

        file_names = [f for f in os.listdir(dataset_dir) if f.endswith(('.pdf', '.jpg', '.jpeg', '.png'))]
        results["total_documents"] = len(file_names)

        for file_name in file_names:
            file_path = os.path.join(dataset_dir, file_name)
            gt_path = os.path.join(ground_truth_dir, os.path.splitext(file_name)[0] + ".json")

            if not os.path.exists(gt_path):
                print(f"[WARN] Ground truth missing for {file_name}, skipping...")
                continue

            with open(gt_path, "r", encoding="utf-8", errors="ignore") as f:
                gt_data = json.load(f)

            start_time = time.time()
            try:
                raw_output = self.model.extract_information(invoice=file_path)
                latency = time.time() - start_time
                pred_data = json.loads(raw_output)
                
                results["schema_compliant_count"] += 1
                results["total_latency_sec"] += latency

                # 1. Header Exact Match
                pred_make = (pred_data.get("car_info", {}).get("car_make") or "").strip().lower()
                gt_make = (gt_data.get("car_info", {}).get("car_make") or "").strip().lower()
                pred_model = (pred_data.get("car_info", {}).get("car_model") or "").strip().lower()
                gt_model = (gt_data.get("car_info", {}).get("car_model") or "").strip().lower()

                if pred_make and gt_make and pred_make == gt_make and pred_model == gt_model:
                    results["header_exact_matches"] += 1

                # 2. Part & Operation Metrics
                metrics = self.calculate_part_metrics(pred_data.get("repaired_parts", []), gt_data.get("repaired_parts", []))
                results["part_precisions"].append(metrics["precision"])
                results["part_recalls"].append(metrics["recall"])
                results["part_f1_scores"].append(metrics["f1_score"])

            except Exception as e:
                print(f"[ERROR] Failure on document {file_name}: {str(e)}")

            # Rate-limiting delay between documents to reduce server overhead
            time.sleep(2.5)

        avg_precision = sum(results["part_precisions"]) / len(results["part_precisions"]) if results["part_precisions"] else 0.0
        avg_recall = sum(results["part_recalls"]) / len(results["part_recalls"]) if results["part_recalls"] else 0.0
        avg_f1 = sum(results["part_f1_scores"]) / len(results["part_f1_scores"]) if results["part_f1_scores"] else 0.0
        avg_latency = results["total_latency_sec"] / results["schema_compliant_count"] if results["schema_compliant_count"] > 0 else 0.0
        schema_accuracy = (results["schema_compliant_count"] / results["total_documents"]) * 100 if results["total_documents"] > 0 else 0.0
        header_em = (results["header_exact_matches"] / results["total_documents"]) * 100 if results["total_documents"] > 0 else 0.0

        return {
            "Model Name": self.model.model_name,
            "Schema Compliance Rate (%)": round(schema_accuracy, 2),
            "Header Exact Match (%)": round(header_em, 2),
            "Part Precision": round(avg_precision, 4),
            "Part Recall": round(avg_recall, 4),
            "Part F1-Score": round(avg_f1, 4),
            "Avg Latency (sec/doc)": round(avg_latency, 2)
        }

In [ ]:
import os
import json
import time

# 1. API Keys initialization
API_KEYS = [
    ""
]

INVOICES_DIR = "/kaggle/input/datasets/nataliemmoris/dataset/invoice-eval-data/invoices"
GROUND_TRUTH_DIR = "/kaggle/input/datasets/nataliemmoris/dataset/invoice-eval-data/ground_truth"
PREDICTIONS_DIR = "/kaggle/working/predictions"

os.makedirs(PREDICTIONS_DIR, exist_ok=True)

# 2. Model & Evaluator initialization
model = InvoiceModel(api_keys=API_KEYS, model_name="gemini-3.6-flash")
evaluator = InvoicePipelineEvaluator(model=model)

# 3. Processing pipeline with incremental caching
file_names = [f for f in os.listdir(INVOICES_DIR) if f.endswith(('.pdf', '.jpg', '.jpeg', '.png'))]
print(f"[INFO] Total Corpus Documents: {len(file_names)}")

for idx, file_name in enumerate(file_names, 1):
    base_name = os.path.splitext(file_name)[0]
    pred_path = os.path.join(PREDICTIONS_DIR, base_name + ".json")
    file_path = os.path.join(INVOICES_DIR, file_name)
    gt_path = os.path.join(GROUND_TRUTH_DIR, base_name + ".json")

    if not os.path.exists(gt_path):
        continue

    # Skip file if already processed and cached to save API quota
    if os.path.exists(pred_path):
        print(f"⏩ Skipping {file_name} (Already Processed and Cached).")
        continue

    print(f"\n[DOCUMENT {idx}/{len(file_names)}] Processing: {file_name}")
    try:
        raw_output = model.extract_information(invoice=file_path)
        pred_data = json.loads(raw_output)
        
        # Save prediction immediately to storage
        with open(pred_path, "w", encoding="utf-8") as f:
            json.dump(pred_data, f, indent=2, ensure_ascii=False)
        print(f"✅ Saved prediction for {file_name}")
        
        time.sleep(2) # Safety rate-limit delay

    except Exception as e:
        print(f"❌ Could not process {file_name}: {str(e)}")

# 4. Offline Metric Evaluation from cached predictions
print("\n" + "="*70)
print("📊 CALCULATING ACCUMULATED METRICS FROM CACHED PREDICTIONS...")
print("="*70)

part_precisions, part_recalls, part_f1_scores = [], [], []
header_matches, compliant_count = 0, 0

processed_files = [f for f in os.listdir(PREDICTIONS_DIR) if f.endswith('.json')]

for p_file in processed_files:
    gt_path = os.path.join(GROUND_TRUTH_DIR, p_file)
    pred_path = os.path.join(PREDICTIONS_DIR, p_file)

    if not os.path.exists(gt_path):

[INFO] Total Corpus Documents: 29
⏩ Skipping 27.pdf (Already Processed and Cached).
⏩ Skipping 40.pdf (Already Processed and Cached).
⏩ Skipping 46.pdf (Already Processed and Cached).
⏩ Skipping 48.pdf (Already Processed and Cached).
⏩ Skipping 53.pdf (Already Processed and Cached).
⏩ Skipping 80.pdf (Already Processed and Cached).
⏩ Skipping Audi_A8L_Invoice.pdf (Already Processed and Cached).
⏩ Skipping Auto Repair Invoice.pdf (Already Processed and Cached).
⏩ Skipping Mitsubishi Xpander.pdf (Already Processed and Cached).
⏩ Skipping Opel_Astra_H_GTC_Invoice_Updated.pdf (Already Processed and Cached).
⏩ Skipping Peugeot_207_Invoice.pdf (Already Processed and Cached).
⏩ Skipping Toyota_Camry_Invoice.pdf (Already Processed and Cached).
⏩ Skipping case1.pdf (Already Processed and Cached).
⏩ Skipping case10.pdf (Already Processed and Cached).
⏩ Skipping case2.pdf (Already Processed and Cached).
⏩ Skipping case3.pdf (Already Processed and Cached).

[DOCUMENT 17/29] Processing: case4.pdf
P